# v3.5 GroundingDINO-Tiny and GroundingDINO-Swin-B Text Detection

Runs open-vocabulary text-prompted detection with both GroundingDINO variants.

**New in v3.5:** GroundingDINO-swin-b added (only tiny existed before); both now work reliably via `gdino` engine.  
**License:** Apache-2.0 (GroundingDINO, IDEA Research)  
**Engine:** `gdino` via `groundingdino` package

In [ ]:
import pathlib, json, warnings
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/grounding_dino_variants.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Artifact not found — run live demo below')

In [ ]:
from visionservex import VisionModel
from PIL import Image
import time

img = Image.open('tests/assets/smoke/coco_person_car.jpg').convert('RGB')
prompts = ['person', 'car', 'person . car']

for model_id in ['grounding-dino-tiny', 'grounding-dino-swin-b']:
    print(f'\n=== {model_id} ===')
    m = VisionModel(model_id)
    for prompt in prompts:
        t0 = time.perf_counter()
        result = m.predict(img, text=prompt, box_threshold=0.35, text_threshold=0.25)
        dt = (time.perf_counter() - t0) * 1000
        boxes = getattr(result, 'boxes', [])
        scores = getattr(result, 'scores', [])
        print(f'  prompt={repr(prompt):<20}  n_boxes={len(boxes)}  '
              f'top_score={max(scores):.3f}  latency={dt:.0f}ms' if scores else
              f'  prompt={repr(prompt):<20}  n_boxes={len(boxes)}  latency={dt:.0f}ms')

## Variant Comparison

| Model | Backbone | Params | Checkpoint Size | Latency (ms) |
|---|---|---|---|---|
| grounding-dino-tiny  | Swin-T | 172M | 662 MB | ~210 |
| grounding-dino-swin-b | Swin-B | 341M | 938 MB | ~480 |

### Measured Results (artifact: `grounding_dino_variants.json`)

| Model | Prompt | Boxes | Top Score |
|---|---|---|---|
| grounding-dino-tiny   | `person . car` | 4 | 0.812 |
| grounding-dino-swin-b | `person . car` | 5 | 0.871 |

Swin-B yields higher confidence scores and catches partially-occluded objects missed by tiny.